Prioritization of ligands based on expression values
================

In this notebook, we will extend the basic NicheNet analysis from [Perform NicheNet analysis: step-by-step analysis](steps.ipynb) by incorporating gene expression as part of the prioritization. While the original NicheNet only ranks ligands based on the ligand activity analysis, it is now also possible to prioritize ligands based on cell type and condition specificity of the ligand and receptor. We will again make use of mouse NICHE-seq data to explore intercellular communication in the T cell area in the inguinal lymph node before and 72 hours after lymphocytic choriomeningitis virus (LCMV) infection (Medaglia et al. 2017). 

Make sure you understand the different steps in a NicheNet analysis that are described in the basic vignette before proceeding with this tutorial.

# Prepare NicheNet analysis

Load required packages, read in the AnnData object with processed expression data of interacting cells and NicheNet’s ligand-target prior model, ligand-receptor network and weighted integrated networks.

In [1]:
from nichenetpy.prediction import LigandActivityPredictor
from nichenetpy.network import LigandReceptorNetwork, WeightedNetwork
from nichenetpy.wrappers import run_nichenet, generate_info_tables
from nichenetpy.utils import read_matrix_from_csv, ligand_activities_df
from nichenetpy.gene_symbol import mouse_alias_info
from nichenetpy.prioritization import (
    generate_prioritization_table,
    process_table_to_ic,
    calculate_de,
    get_avg_exp
)
from nichenetpy.metrics import group_metrics

import anndata
import pandas as pd

In [2]:
ann = anndata.io.read_h5ad("D:/Data/nichenetpy/annData/annData3531889.h5")
ann.var_names = ann.var["gene"]
mouse_alias_info.alias_to_symbol(ann)
predictor = LigandActivityPredictor(*read_matrix_from_csv("D:/Data/nichenetpy/model/mouse/csv/ligand_target_matrix_mouse.csv"))
lr_network = LigandReceptorNetwork(filename="D:/Data/nichenetpy/model/mouse/csv/lr_network_mouse.csv")
lr_sig = WeightedNetwork(filename="D:/Data/nichenetpy/model/mouse/csv/lr_sig_mouse.csv")

# Perform the NicheNet analysis

We will use the sender-focused approach here.

In [3]:
sender_celltypes = ("CD4 T", "Treg", "Mono", "NK", "B", "DC")
res = run_nichenet(
    ann,
    predictor,
    lr_network,
    "CD8 T",
    "LCMV",
    "SS",
    sender_celltypes=sender_celltypes,
)
ligand_activities_sorted = res["ligand_activities_sorted_focused"]
best_upstream_ligands = res["best_upstream_ligands_focused"]
expressed_ligands = res["expressed_ligands"]
expressed_receptors = res["expressed_receptors"]

# Perform prioritization of ligand-receptor pairs

We will prioritize ligand-receptor pairs based on the following criteria
(with their corresponding weight names):

- Upregulation of the ligand in a sender cell type compared to other
  cell types: `de_ligand`
- Upregulation of the receptor in a receiver cell type: `de_receptor`
- Average expression of the ligand in the sender cell type:
  `exprs_ligand`
- Average expression of the receptor in the receiver cell type:
  `exprs_receptor`
- Condition-specificity of the ligand across all cell types:
  `ligand_condition_specificity`
- Condition-specificity of the receptor across all cell types:
  `receptor_condition_specificity`

This means that we will have to calculate:

- Differential expression of the ligand/receptor in a sender/receiver
  cell type
- The average expression of each ligand/receptor in each sender/receiver
  cell type
- Differential expression of the ligand/receptor between the two
  conditions

We provide a wrapper function `generate_info_tables` that will calculate
all these values for you. This function returns a list with three
dataframes:

- `sender_receiver_de`: differential expression of the ligand and
  receptor in the sender-receiver cell type pair. These were first
  calculated separately (i.e., DE of ligand in sender cell type, DE of
  receptor in receiver cell type based on FindAllMarkers) and then
  combined based on possible interactions from the lr_network.
- `sender_receiver_info`: the average expression of the ligand and
  receptor in sender-receiver cell type pairs
- `lr_condition_de`: differential expression of the ligand and receptor
  between the two conditions across all cell types.

Note that cell type specificity (i.e., the first four conditions) is
calculated only in the condition of interest.

The “scenario” argument can be either “case_control” or “one_condition”.
In “case_control” scenario, condition specificity is calculated.

In [4]:
lr_network_filtered = lr_network.subset_sep(expressed_ligands, expressed_receptors)

In [5]:
info_tables = generate_info_tables(
    ann,
    "celltype",
    sender_celltypes,
    ["CD8 T"],
    lr_network_filtered,
    "aggregate",
    "LCMV",
    "SS",
    case_control=True
)

Next, we generate the prioritization table. This table contains the rankings of ligand-receptor pairs based on the different criteria. We provide two scenarios: case_control and one_condition. In the “case_control” scenario, all weights are set to 1. If “one_condition”, the weights are set to 0 for condition specificity and 1 for the remaining criteria. Users can also provide their own weights using the prioritizing_weights argument.

In [6]:
prior_table = generate_prioritization_table(
    info_tables["sender_receiver_info"],
    info_tables["sender_receiver_de"],
    ligand_activities_sorted,
    info_tables["lr_condition_de"]
)

536     1.017174e-272
140     1.590801e-296
1214    2.637179e-194
296     1.138617e-243
405      2.996834e-57
            ...      
266      5.586557e-01
931      4.274726e-03
458      1.369870e-01
409      8.033137e-04
746      1.766171e-02
Name: pval_ligand, Length: 732, dtype: float64
536     5.250531e-206
542      6.104465e-17
140      6.628900e-04
1214     1.470347e-02
134      5.070025e-05
            ...      
161      8.074156e-01
1125     1.074424e-01
942      3.371005e-01
354      1.661213e-01
417      6.266626e-01
Name: pval_receptor, Length: 66, dtype: float64
815    6.631501e-25
487    5.189217e-09
189    1.621364e-04
696    6.820290e-04
233    2.996288e-02
           ...     
587    1.284697e-02
698    6.820290e-04
191    1.621364e-04
485    5.189217e-09
813    6.631501e-25
Name: pval_ligand, Length: 244, dtype: float64
815    1.819126e-04
487    1.120017e-01
814    1.819126e-04
482    8.562515e-01
483    8.562515e-01
           ...     
239    1.108254e-02
546    7.08116

c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [7]:
prior_table[["sender", "receiver", "ligand", "receptor", "prioritization_score"]].drop_duplicates()

,sender,receiver,ligand,receptor,prioritization_score
255,Mono,CD8 T,Il27,Il27ra,0.386153
407,Mono,CD8 T,Ebi3,Il27ra,0.386153
395,Mono,CD8 T,Ebi3,Il6st,0.363837
491,Treg,CD8 T,Ebi3,Il27ra,0.349493
1587,NK,CD8 T,Ptprc,Dpp4,0.332220
...,...,...,...,...,...
4708,CD4 T,CD8 T,Mif,Cd74,0.023215
2845,DC,CD8 T,Calr,Tap1,0.021603
3166,Mono,CD8 T,Mif,Cxcr4,0.016403
4664,DC,CD8 T,Mif,Cd74,0.011268


### step-by-step prioritization

In [8]:
sender_receiver_de = process_table_to_ic(
    calculate_de(
        ann,
        "celltype",
        "LCMV",
        "aggregate",
        features=lr_network_filtered.get_ligands().union(lr_network_filtered.get_receptors())
    ),
    "celltype_DE",
    lr_network_filtered,
    sender_celltypes,
    ["CD8 T"]
)
sender_receiver_info = process_table_to_ic(
    get_avg_exp(
        ann,
        "celltype",
        "LCMV",
        "aggregate"
    ),
    "expression",
    lr_network_filtered
)
group_metrics(
    ann,
    groupby="aggregate"
)
res = ann.uns["group_metrics"]
lr_condition_de = process_table_to_ic(
    res[["gene", "lfc", "pval", "pval_adj"]],
    "group_DE",
    lr_network_filtered
)

And here is how you can define custom weights:

In [9]:
prioritizing_weights = {
    "de_ligand": 1,
    "de_receptor": 1,
    "activity_scaled": 1,
    "exprs_ligand": 1,
    "exprs_receptor": 1,
    "ligand_condition_specificity": 1,
    "receptor_condition_specificity": 1
}
prior_table = generate_prioritization_table(
    sender_receiver_info,
    sender_receiver_de,
    ligand_activities_sorted,
    lr_condition_de
)

536     1.017174e-272
140     1.590801e-296
1214    2.637179e-194
296     1.138617e-243
405      2.996834e-57
            ...      
266      5.586557e-01
931      4.274726e-03
458      1.369870e-01
409      8.033137e-04
746      1.766171e-02
Name: pval_ligand, Length: 732, dtype: float64
536     5.250531e-206
542      6.104465e-17
140      6.628900e-04
1214     1.470347e-02
134      5.070025e-05
            ...      
161      8.074156e-01
1125     1.074424e-01
942      3.371005e-01
354      1.661213e-01
417      6.266626e-01
Name: pval_receptor, Length: 66, dtype: float64
815    6.631501e-25
487    5.189217e-09
189    1.621364e-04
696    6.820290e-04
233    2.996288e-02
           ...     
587    1.284697e-02
698    6.820290e-04
191    1.621364e-04
485    5.189217e-09
813    6.631501e-25
Name: pval_ligand, Length: 244, dtype: float64
815    1.819126e-04
487    1.120017e-01
814    1.819126e-04
482    8.562515e-01
483    8.562515e-01
           ...     
239    1.108254e-02
546    7.08116

c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


## Prioritizing across multiple receivers

As NicheNet is a receiver-based pipeline, to prioritize ligand-receptor pairs across multiple receivers, we need to perform the NicheNet analysis for each receiver separately. Let’s suppose we want to prioritize ligand-receptor pairs across all T cells (CD4, CD8, and Tregs). The CD8 T analysis has already been performed above. We will use the wrapper function to perform a basic NicheNet analysis on the other two:

In [10]:
nichenet_output = dict()
for receiver in ("CD8 T", "CD4 T", "Treg"):
    res = run_nichenet(
        ann,
        predictor,
        lr_network,
        receiver,
        "LCMV",
        "SS",
        sender_celltypes=sender_celltypes,
    )
    ligand_activities = ligand_activities_df(res["ligand_activities_sorted_focused"])
    ligand_activities["receiver"] = [receiver for _ in range(len(ligand_activities.index))]
    res["ligand_activities"] = ligand_activities
    nichenet_output[receiver] = res


To generate the dataframes used for prioritization, we will simply change the lr_network_filtered argument to only calculate DE and expression values for ligand-receptor pairs of interest.

In [11]:
for receiver, res in nichenet_output.items():
    lr_network_filtered = lr_network.subset_sep(
        res["ligand_activities"].index,
        [gene for gene in res["expressed_genes_receiver"] if gene in predictor.row_names]
    )
    res["info_tables"] = generate_info_tables(
        ann,
        "celltype",
        sender_celltypes,
        [receiver],
        lr_network_filtered,
        "aggregate",
        "LCMV",
        "SS",
        case_control=True
    )
info_tables_combined = {
    key: pd.concat((res["info_tables"][key] for res in nichenet_output.values()))
    for key in ["sender_receiver_de", "sender_receiver_info", "lr_condition_de"]
}
info_tables_combined["sender_receiver_info"].drop_duplicates(inplace=True)
info_tables_combined["lr_condition_de"].drop_duplicates(inplace=True)
ligand_activities_combined = pd.concat((res["ligand_activities"] for res in nichenet_output.values()))
prior_tables_combined = generate_prioritization_table(
    info_tables_combined["sender_receiver_info"],
    info_tables_combined["sender_receiver_de"],
    ligand_activities_combined,
    info_tables_combined["lr_condition_de"]
)

KeyboardInterrupt: 

### Extra visualization of ligand-receptor pairs

In addition to the usual heatmap visualizations, we provide a function make_circos_lr to visualize the ligand-receptor pairs in a circos plot. This was originally written for the (now deprecated) Differential NicheNet vignettes. The function takes in a prioritization table and a named vector for the color of senders and receivers. We first specify the number of top ligand-receptor pairs to show with n.

In [12]:
# TODO